In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

In [4]:
df_pedidos = pd.read_csv('pedidos_historicos_genericos.csv')
df_pronostico = pd.read_csv('pronostico_demanda_generico.csv')
df_capacidad = pd.read_csv('capacidad_logistica_generica.csv')
df_plan = pd.read_csv('plan_entrega_clientes_generico.csv')

df_pedidos['fecha_despacho'] = pd.to_datetime(df_pedidos['fecha_despacho'])
df_pronostico['fecha_despacho'] = pd.to_datetime(df_pronostico['fecha_despacho'])
df_pronostico['fecha_emision'] = pd.to_datetime(df_pronostico['fecha_emision'])
df_plan['fecha_plan'] = pd.to_datetime(df_plan['fecha_plan'])
df_plan['fecha_compra_real'] = pd.to_datetime(df_plan['fecha_compra_real'])

df_reales = df_pedidos.groupby(['centro_distribucion', 'fecha_despacho'], as_index=False)['unidades_reales'].sum()

## Analisis impacto operativo

In [6]:

df_cap_merge = pd.merge(
    df_reales,
    df_capacidad,
    left_on=['centro_distribucion', 'fecha_despacho'],
    right_on=['centro_distribucion', 'fecha'],
    how='inner'
)

df_cap_merge['exceso_capacidad'] = np.maximum(0, df_cap_merge['unidades_reales'] - df_cap_merge['capacidad_maxima_unidades'])
df_cap_merge['capacidad_ociosa'] = np.maximum(0, df_cap_merge['capacidad_maxima_unidades'] - df_cap_merge['unidades_reales'])

resumen_capacidad = df_cap_merge.groupby('centro_distribucion').agg(
    capacidad_promedio=('capacidad_maxima_unidades', 'mean'),
    demanda_real_promedio=('unidades_reales', 'mean'),
    dias_sobrepasa_capacidad=('exceso_capacidad', lambda x: (x > 0).sum()),
    total_unidades_exceso=('exceso_capacidad', 'sum'),
    total_unidades_ociosas=('capacidad_ociosa', 'sum')
).reset_index()

print("--- Resumen de Capacidad Logística por CD ---")
print(resumen_capacidad)

--- Resumen de Capacidad Logística por CD ---
  centro_distribucion  capacidad_promedio  demanda_real_promedio  \
0           CD_CENTRO              1860.0            1510.433500   
1             CD_ESTE               992.0             796.946167   
2            CD_NORTE              3720.0            3074.746333   
3            CD_OESTE               620.0             489.994333   
4         CD_PACIFICO               806.0             664.160333   
5              CD_SUR              2728.0            2221.993667   

   dias_sobrepasa_capacidad  total_unidades_exceso  total_unidades_ociosas  
0                         2                 122.21                21096.20  
1                         1                  70.04                11773.27  
2                         6                 974.26                39689.48  
3                         1                  11.28                 7811.62  
4                         1                   8.14                 8518.52  
5              

# Comportamiento por canal de venta

In [7]:
df_pedidos['dia_semana'] = df_pedidos['fecha_despacho'].dt.day_name()
df_pedidos['num_dia'] = df_pedidos['fecha_despacho'].dt.dayofweek

df_canal = df_pedidos.groupby(['canal_venta', 'dia_semana', 'num_dia'], as_index=False)['unidades_reales'].sum()
pivote_canal = df_canal.pivot(index='canal_venta', columns='dia_semana', values='unidades_reales').fillna(0)

dias_ordenados = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivote_canal = pivote_canal.reindex(columns=[d for d in dias_ordenados if d in pivote_canal.columns])

print("--- Distribución de Ventas por Canal y Día de la Semana ---")
print(pivote_canal)

df_plan = pd.read_csv('plan_entrega_clientes_generico.csv')
fuera_plan_canal = df_plan.groupby('canal_venta')['fuera_de_plan'].agg(
    total_registros='count',
    pedidos_fuera='sum',
    porcentaje=lambda x: x.mean() * 100
).reset_index()

print("\n--- Porcentaje Fuera de Plan por Canal de Venta ---")
print(fuera_plan_canal)

--- Distribución de Ventas por Canal y Día de la Semana ---
dia_semana             Monday   Tuesday  Wednesday  Thursday    Friday  \
canal_venta                                                              
AGENTE_COMERCIAL     18496.14  23940.75   22178.29  23993.68  28948.84   
ATENCION_TELEFONICA  13652.91  19073.48   13759.07  13343.78  13250.09   
PORTAL_WEB           36363.84  44372.90   33872.01  40262.61  37120.51   

dia_semana           Saturday    Sunday  
canal_venta                              
AGENTE_COMERCIAL     21021.69  17814.86  
ATENCION_TELEFONICA  15655.50  11738.36  
PORTAL_WEB           40725.90  35911.25  

--- Porcentaje Fuera de Plan por Canal de Venta ---
           canal_venta  total_registros  pedidos_fuera  porcentaje
0     AGENTE_COMERCIAL              173             29   16.763006
1  ATENCION_TELEFONICA              131             21   16.030534
2           PORTAL_WEB              435             79   18.160920
